# UrbanOS text triage

Fine-tune Qwen2.5-0.5B-Instruct with LoRA on a Google-hosted GPU. Evaluate on synthetic complaints with held-out wordings and localities, then optionally save the adapter to Hugging Face and deploy it to a Docker Space.

Open this notebook in Colab. Select **Runtime → Change runtime type → GPU**, then run sections 1–4b in order. Keep the accompanying `UrbanOS_Colab_Source.zip` ready for cell 2.

The source bundle contains the complaint generator and training code. The model learns from generated complaints. Cell 4 reports synthetic validation scores for routing, urgency, category, issue tags and zones. Voice and photo intake are outside this model's scope.

## 1. Prepare the hosted GPU runtime

In [ ]:
from pathlib import Path
import os
import subprocess
import sys
import tempfile

from google.colab import files, userdata
import torch

if not Path("/content").is_dir() or not torch.cuda.is_available():
    raise RuntimeError("Connect to a Google-hosted Colab GPU runtime first.")

WORK = Path(tempfile.mkdtemp(prefix="urbanos-", dir="/content"))
SOURCE = WORK / "source"
SOURCE.mkdir()
BASE = "Qwen/Qwen2.5-0.5B-Instruct"
BASE_REVISION = "7ae557604adf67be50417f59c2c2f167def9a775"
GGUF_REPO = "Qwen/Qwen2.5-0.5B-Instruct-GGUF"
GGUF_REVISION = "9217f5db79a29953eb74d5343926648285ec7e67"
GGUF_NAME = "qwen2.5-0.5b-instruct-q4_k_m.gguf"
MODEL_ALIAS = "urbanos-qwen-triage"
LLAMA_RELEASE = "b12287"
LLAMA_SHA256 = "c0f668e710010e273d75634af937500aef4d19834edfed2d8b2608233c3a991a"

def run(*command, cwd=WORK, env=None):
    subprocess.run([str(part) for part in command], cwd=cwd, env=env, check=True)

precision = "--bf16-base" if torch.cuda.get_device_capability(0)[0] >= 8 else "--fp16-base"
print(f"GPU {torch.cuda.get_device_name(0)} · {precision}")
run(
    sys.executable, "-m", "pip", "install", "--quiet",
    "transformers==4.57.6", "peft==0.15.2", "accelerate==1.6.0",
    "datasets==3.5.0", "huggingface_hub==0.36.2", "sentencepiece",
    "pydantic>=2,<3", "httpx", "python-dotenv",
)

## 2. Upload the training source bundle

In [ ]:
from zipfile import ZipFile

source_files = [
    "llm.py",
    "triage.py",
    "triage_rules.py",
    "finetune/__init__.py",
    "finetune/build_dataset.py",
    "finetune/build_open_dataset.py",
    "finetune/labels.py",
    "finetune/phrasings.py",
    "finetune/score_holdout.py",
    "finetune/train_lora.py",
    "finetune/hf_space/Dockerfile",
    "finetune/hf_space/README.md"
]
bundle = WORK / "UrbanOS_Colab_Source.zip"
files.upload_file(str(bundle))

with ZipFile(bundle) as archive:
    if sorted(archive.namelist()) != sorted(source_files):
        raise ValueError("Use the UrbanOS source bundle provided with this notebook.")
    for name in source_files:
        path = SOURCE / name
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_bytes(archive.read(name))

sys.path.insert(0, str(SOURCE))
print(f"Loaded {len(source_files)} source files.")

## 3. Build the data and train the LoRA adapter

In [ ]:
from huggingface_hub import snapshot_download

base = Path(snapshot_download(
    BASE, revision=BASE_REVISION, token=False,
    allow_patterns=["*.json", "*.safetensors", "merges.txt", "LICENSE"],
))
run_dir = Path(tempfile.mkdtemp(prefix="run-", dir=WORK))
data = run_dir / "data"
adapter = run_dir / "adapter"
run(sys.executable, "-m", "finetune.build_open_dataset", "--out", data, cwd=SOURCE)
run(
    sys.executable, "-m", "finetune.train_lora",
    "--base", base, precision,
    "--train", data / "train.jsonl", "--val", data / "val.jsonl",
    "--out", adapter, "--max-seq-len", "1024",
    "--epochs", "4", "--batch", "4", "--accum", "2", "--lr", "0.0004",
    cwd=SOURCE,
)

## 4. Export and evaluate with the serving runtime

The test server runs inside the Colab VM. The evaluation records classification accuracy by field and counts inference errors.

In [ ]:
import hashlib
import json
import shutil
import time

import httpx
from huggingface_hub import hf_hub_download

tools_dir = WORK / "llama-tools"
llama_source = tools_dir / "llama.cpp"
llama_bin = tools_dir / "bin"
server = llama_bin / "llama-server"
tools_dir.mkdir(exist_ok=True)
if not llama_source.is_dir():
    run("git", "clone", "--depth", "1", "--branch", LLAMA_RELEASE,
        "https://github.com/ggml-org/llama.cpp.git", llama_source)
if not server.is_file():
    filename = f"llama-{LLAMA_RELEASE}-bin-ubuntu-x64.tar.gz"
    archive = tools_dir / filename
    url = f"https://github.com/ggml-org/llama.cpp/releases/download/{LLAMA_RELEASE}/{filename}"
    run("curl", "-fsSL", "--retry", "3", url, "-o", archive)
    with archive.open("rb") as stream:
        if hashlib.file_digest(stream, "sha256").hexdigest() != LLAMA_SHA256:
            raise RuntimeError("llama.cpp release checksum mismatch.")
    llama_bin.mkdir(exist_ok=True)
    run("tar", "-xzf", archive, "--strip-components=1", "-C", llama_bin)
    archive.unlink()

artifact = run_dir / "model"
artifact.mkdir(exist_ok=True)
adapter_gguf = artifact / "adapter.gguf"
run(sys.executable, llama_source / "convert_lora_to_gguf.py", "--base", base,
    "--outtype", "f16", "--outfile", adapter_gguf, adapter)
gguf = hf_hub_download(GGUF_REPO, GGUF_NAME, revision=GGUF_REVISION, token=False)

host = "http://127.0.0.1:8082"
log_path = run_dir / "llama-server.log"
env = {**os.environ, "LLAMA_CPP_HOST": host, "LLAMA_CPP_API_TOKEN": ""}
env.pop("LLAMA_API_KEY", None)
with log_path.open("w") as log:
    process = subprocess.Popen(
        [str(server), "--model", gguf, "--lora", str(adapter_gguf), "--alias", MODEL_ALIAS,
         "--host", "127.0.0.1", "--port", "8082", "--threads", "2", "--ctx-size", "2048",
         "--parallel", "1", "--n-gpu-layers", "0", "--no-webui"],
        stdout=log, stderr=subprocess.STDOUT, env=env,
    )
    try:
        ready = False
        for _ in range(60):
            if process.poll() is not None:
                break
            try:
                if httpx.get(host + "/health", timeout=2).status_code == 200:
                    ready = True
                    break
            except httpx.RequestError:
                pass
            time.sleep(1)
        if not ready:
            raise RuntimeError(f"Serving runtime failed to load the adapter. See {log_path}.")
        run(sys.executable, "-m", "finetune.score_holdout", "--llama-cpp", "--model", MODEL_ALIAS,
            "--val", data / "val.jsonl", "--json", artifact / "evaluation.json", cwd=SOURCE, env=env)
    finally:
        if process.poll() is None:
            process.terminate()
            try:
                process.wait(timeout=10)
            except subprocess.TimeoutExpired:
                process.kill()
                process.wait()

evaluation = json.loads((artifact / "evaluation.json").read_text())
if not evaluation["scored"]:
    raise RuntimeError("No held-out cases completed inference. Check the evaluation log above.")
shutil.copy2(adapter / "adapter_model.safetensors", artifact)
shutil.copy2(base / "LICENSE", artifact)
config = json.loads((adapter / "adapter_config.json").read_text())
config.update(base_model_name_or_path=BASE, revision=BASE_REVISION)
(artifact / "adapter_config.json").write_text(json.dumps(config, indent=2))
report = json.loads((adapter / "train_report.json").read_text())
report.update(base=BASE, base_revision=BASE_REVISION, data="synthetic complaints")
(artifact / "train_report.json").write_text(json.dumps(report, indent=2))
card = f"""---
base_model: {BASE}
base_model_relation: adapter
license: apache-2.0
library_name: peft
tags:
- lora
- civic-triage
---

# UrbanOS text triage adapter

LoRA adapter for classifying civic complaints using the UrbanOS triage schema.
Training used {report["train_examples"]} synthetic examples. See
`evaluation.json` for held-out scores and error counts.
"""
(artifact / "README.md").write_text(card)
print(f"Model artifacts ready at {artifact}")

## 4b. Compare with the baselines

A score means little on its own. This cell scores the same 26 held-out complaints two more ways, using the same scorer: UrbanOS's keyword rules, which run when no model is reachable, and the base Qwen model before fine-tuning. The fine-tuned model should beat both.

In [ ]:
from llm import LlamaCppProvider
import re

from finetune import score_holdout
import triage_rules


class RulesProvider:
    """UrbanOS's keyword fallback, shaped like a model's reply so the same scorer applies."""

    def classify(self, prompt, schema, image=None):
        text = re.search(r"Complaint Text: (.*?)(?:\nSubmitted Location:|$)", prompt, re.S).group(1).strip()
        found = re.search(r"Submitted Location: (.*)", prompt)
        location = found.group(1).strip() if found else ""
        r = triage_rules.classify(text, location)
        return schema.model_validate({
            "category": r["category"], "priority": r["priority"], "sentiment": r["feasibility"],
            "extracted_location": location, "summary": r["summary"],
            "original_language": r["original_language"], "constituency_zone": r["constituency_zone"],
            "estimated_budget": r["estimated_budget"], "responsible_body": r["responsible_body"],
            "semantic_tag": r["semantic_tag"], "visual_evidence": "",
        })


val_path = data / "val.jsonl"
results = {"keyword rules": score_holdout.score("keyword-rules", val_path, provider=RulesProvider())}

# The base model, served exactly like the fine-tuned one but without the adapter.
base_log = run_dir / "llama-server-base.log"
with base_log.open("w") as log:
    base_process = subprocess.Popen(
        [str(server), "--model", gguf, "--alias", MODEL_ALIAS,
         "--host", "127.0.0.1", "--port", "8082", "--threads", "2", "--ctx-size", "2048",
         "--parallel", "1", "--n-gpu-layers", "0", "--no-webui"],
        stdout=log, stderr=subprocess.STDOUT, env=env,
    )
try:
    for _ in range(60):
        if base_process.poll() is not None:
            raise RuntimeError(f"The base model failed to start. See {base_log}.")
        try:
            if httpx.get(host + "/health", timeout=2).status_code == 200:
                break
        except httpx.RequestError:
            pass
        time.sleep(1)
    else:
        raise RuntimeError(f"The base model did not become ready. See {base_log}.")
    os.environ["LLAMA_CPP_HOST"], os.environ["LLAMA_CPP_API_TOKEN"] = host, ""
    results["base Qwen, not fine-tuned"] = score_holdout.score(
        MODEL_ALIAS, val_path, provider=LlamaCppProvider(host=host, model=MODEL_ALIAS))
finally:
    if base_process.poll() is None:
        base_process.terminate()
        base_process.wait(timeout=10)

results["fine-tuned Qwen"] = json.loads((artifact / "evaluation.json").read_text())

fields = ["responsible_body", "category", "priority", "semantic_tag", "constituency_zone"]
names = {"responsible_body": "Department", "category": "Category", "priority": "Priority",
         "semantic_tag": "Issue tag", "constituency_zone": "Zone"}
print(f"{'':28s}" + "".join(f"{names[f]:>12s}" for f in fields) + f"{'Errors':>9s}")
for label, r in results.items():
    print(f"{label:28s}" + "".join(f"{r['accuracy'][f]:>12.0%}" for f in fields) + f"{r['errors']:>9d}")

comparison = {label: {"accuracy": r["accuracy"], "scored": r["scored"], "errors": r["errors"]}
              for label, r in results.items()}
(artifact / "comparison.json").write_text(json.dumps(comparison, indent=2))
print(f"\nSaved {artifact / 'comparison.json'}")

## 5. Save the adapter to Hugging Face

Add a Hugging Face token with repository write access in Colab's **Secrets** panel as `HF_TOKEN`. Enable notebook access to it. Run this cell to upload the adapter and evaluation to a private model repository. The cell uses your Hugging Face username as the default namespace.

In [ ]:
from huggingface_hub import HfApi

model_name = "urbanos-qwen-triage"
api = HfApi(token=userdata.get("HF_TOKEN"))
namespace = api.whoami()["name"]
model_id = f"{namespace}/{model_name}"

api.create_repo(model_id, repo_type="model", private=True, exist_ok=True)
if not api.model_info(model_id).private:
    raise RuntimeError("Choose a new model name or make the existing repository private.")
model_commit = api.upload_folder(
    repo_id=model_id, repo_type="model", folder_path=artifact,
    allow_patterns=["README.md", "LICENSE", "adapter.gguf", "adapter_model.safetensors",
                    "adapter_config.json", "train_report.json", "evaluation.json", "comparison.json"],
    commit_message="Upload trained UrbanOS triage adapter and held-out results",
)
print(model_commit.repo_url)

## 6. Create a Hugging Face Docker Space

This step requires a Hugging Face plan that supports Docker Spaces. It uses CPU Basic. Add a separate, random `LLAMA_API_KEY` in Colab Secrets and enable notebook access. This is the endpoint credential. Run this cell to create a private Space and upload its Dockerfile and adapter.

In [ ]:
space_id = f"{namespace}/urbanos-triage-api"
key = userdata.get("LLAMA_API_KEY").strip()
if not key or "," in key:
    raise ValueError("Use a single random string for LLAMA_API_KEY.")

space_url = api.create_repo(
    space_id, repo_type="space", space_sdk="docker", private=True, exist_ok=True,
)
info = api.space_info(space_id)
if not info.private or info.sdk not in {None, "docker"}:
    raise RuntimeError("Choose the name of a private Docker Space.")
api.add_space_secret(space_id, "LLAMA_API_KEY", key)

space_files = run_dir / "space"
space_files.mkdir(exist_ok=True)
for name in ("README.md", "Dockerfile"):
    shutil.copy2(SOURCE / "finetune" / "hf_space" / name, space_files)
shutil.copy2(adapter_gguf, space_files)
api.upload_folder(
    repo_id=space_id, repo_type="space", folder_path=space_files,
    allow_patterns=["README.md", "Dockerfile", "adapter.gguf"],
    commit_message="Deploy UrbanOS triage adapter",
)
print(f"Space settings: {space_url}/settings")

## 7. Verify the hosted API

In the Space settings, select **Protected** visibility. This keeps the repository private and enables an HTTPS endpoint for the backend. The model endpoint requires the `LLAMA_API_KEY` bearer token.

Run this cell to wait for the Space, check that an unauthenticated completion is rejected, and classify a synthetic complaint using the same provider as UrbanOS.

In [ ]:
from llm import LlamaCppProvider
from triage import TriageResult

deadline = time.monotonic() + 420
endpoint = None
while time.monotonic() < deadline:
    info = api.space_info(space_id)
    runtime = api.get_space_runtime(space_id)
    if runtime.stage in {"BUILD_ERROR", "RUNTIME_ERROR", "CONFIG_ERROR"}:
        raise RuntimeError(f"Space is in {runtime.stage}. Open its logs for details.")
    candidate = info.host or (f"https://{info.subdomain}.hf.space" if info.subdomain else None)
    if candidate:
        try:
            if httpx.get(candidate.rstrip("/") + "/health", timeout=5).status_code == 200:
                endpoint = candidate.rstrip("/")
                break
        except httpx.RequestError:
            pass
    time.sleep(10)

if endpoint is None:
    raise RuntimeError("Space did not become reachable. Check its build status and Protected visibility, then rerun this cell.")

sample = json.loads((data / "val.jsonl").read_text(encoding="utf-8").splitlines()[0])
unauthenticated = httpx.post(endpoint + "/v1/chat/completions", timeout=15, json={
    "model": MODEL_ALIAS,
    "messages": [{"role": "user", "content": "Reply OK"}],
    "max_tokens": 2,
})
if unauthenticated.status_code not in {401, 403}:
    raise RuntimeError(f"Expected the unauthenticated request to fail with 401 or 403, got {unauthenticated.status_code}.")

previous_key = os.environ.get("LLAMA_CPP_API_TOKEN")
os.environ["LLAMA_CPP_API_TOKEN"] = key
try:
    result = LlamaCppProvider(host=endpoint, model=MODEL_ALIAS).classify(
        sample["messages"][0]["content"], TriageResult
    )
finally:
    if previous_key is None:
        os.environ.pop("LLAMA_CPP_API_TOKEN", None)
    else:
        os.environ["LLAMA_CPP_API_TOKEN"] = previous_key

print(json.dumps(result.model_dump(by_alias=True), ensure_ascii=False, indent=2))
print(f"LLAMA_CPP_HOST={endpoint}")
print(f"LLAMA_CPP_MODEL={MODEL_ALIAS}")

## Connect UrbanOS

In the backend environment set `LLM_PROVIDER=llama_cpp`, `LLAMA_CPP_HOST` to the verified Space URL and `LLAMA_CPP_MODEL=urbanos-qwen-triage`. Set the backend secret `LLAMA_CPP_API_TOKEN` to the same value as the Space's `LLAMA_API_KEY`. The browser does not need the token.

CPU Basic can sleep after inactivity. Use the Space as a prototype endpoint and measure latency and throughput before choosing production serving capacity.

[Colab runtime FAQ](https://research.google.com/colaboratory/faq.html) · [Hugging Face Spaces plans and visibility](https://huggingface.co/docs/hub/en/spaces-overview)